In [1]:
import pandas as pd
import sys
import os
from pathlib import Path
from datetime import datetime

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.features.features_v1 import *
from src.utils.helper_functions import *
from src.utils.team_info import teamStarPlayer, projectedStartingFive, mainStartingFive
from src.analysis.poissonFunctions import (
    compute_bayesian_lambda,
    compute_bayesian_lambda_assists,
    compute_bayesian_lambda_rebounds,
    compute_bayesian_lambda_blocks,
    compute_bayesian_lambda_steals
)

from scipy.stats import poisson
import numpy as np
from nba_api.stats.endpoints import leaguedashteamstats

In [2]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

if dfs_file is None:
    raise FileNotFoundError(f"No NBA_DFS file found for {today}")

s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
s26.rename(columns={'BLK_x': 'BLK'}, inplace=True)
dfsData = pd.read_csv(dfs_file)

print(f"Loaded: {dfs_file.name}")
dfsData.head()

Loaded: NBA_DFS_20251207_104754.csv


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Betr DFS,player_points,Josh Hart,Over,18.5,-137,2025-12-07,2025-12-07T18:47:15Z,2025-12-07 10:47:54
1,Betr DFS,player_points,Josh Hart,Under,18.5,-137,2025-12-07,2025-12-07T18:47:15Z,2025-12-07 10:47:54
2,Betr DFS,player_points,Mikal Bridges,Over,14.5,-137,2025-12-07,2025-12-07T18:47:15Z,2025-12-07 10:47:54
3,Betr DFS,player_points,Mikal Bridges,Under,14.5,-137,2025-12-07,2025-12-07T18:47:15Z,2025-12-07 10:47:54
4,Betr DFS,player_points,Jaylen Brown,Over,29.5,-137,2025-12-07,2025-12-07T18:47:15Z,2025-12-07 10:47:54


## Points

### prizepicks

In [3]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]
res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_off_rtg = league_df['OFF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_pts = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_off_rtg, league_avg_def_rtg,
        league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_pts % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_pts) + 1)
    elif target_pts % 1 == 0:
        prob_over_poisson = poisson.sf(target_pts, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_pts), int(target_pts) + 1)
    else:
        # Handle other cases
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_pts,
        'L-5': round(count_line_hits(player_df, target_pts, 'player_points', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_pts, 'player_points', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_pts, 'player_points', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })

point_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
point_df.to_csv(f'data/props/prizepicks/player_points.csv', index=False)
point_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Austin Reaves,23.0,0.8,0.8,0.73,0.934,0.066
1,Jaylen Wells,12.5,1.0,0.7,0.53,0.828,0.172
2,Jamal Murray,23.5,0.6,0.6,0.40,0.812,0.188
3,Coby White,19.5,1.0,0.6,0.40,0.799,0.201
4,Jordan Walsh,7.5,0.8,0.5,0.40,0.772,0.228
5,Jaylen Brown,28.5,0.8,0.7,0.67,0.757,0.243
6,Jake LaRavia,4.5,0.6,0.6,0.73,0.755,0.245
7,Cam Spencer,11.5,0.6,0.6,0.53,0.752,0.248
8,Joel Embiid,17.5,0.8,0.6,0.40,0.749,0.251
9,VJ Edgecombe,10.0,0.4,0.6,0.67,0.748,0.252


### underdog

In [4]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]
res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_off_rtg = league_df['OFF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_pts = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_off_rtg, league_avg_def_rtg,
        league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_pts % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_pts) + 1)
    elif target_pts % 1 == 0:
        prob_over_poisson = poisson.sf(target_pts, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_pts), int(target_pts) + 1)
    else:
        # Handle other cases
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_pts,
        'L-5': round(count_line_hits(player_df, target_pts, 'player_points', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_pts, 'player_points', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_pts, 'player_points', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })

point_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
point_df.to_csv(f'data/props/underdog/player_points.csv', index=False)
point_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Austin Reaves,23.5,0.8,0.8,0.73,0.934,0.066
1,Jaylen Wells,12.5,1.0,0.7,0.53,0.828,0.172
2,Jared McCain,8.5,0.4,0.4,0.27,0.778,0.222
3,Jordan Walsh,7.5,0.8,0.5,0.40,0.772,0.228
4,Cam Spencer,11.5,0.6,0.6,0.53,0.752,0.248
5,Quentin Grimes,13.5,0.6,0.7,0.60,0.751,0.249
6,Joel Embiid,17.5,0.8,0.6,0.40,0.749,0.251
7,VJ Edgecombe,10.5,0.4,0.6,0.67,0.748,0.252
8,Jerami Grant,18.5,0.4,0.6,0.47,0.735,0.265
9,Sam Hauser,6.5,0.6,0.5,0.33,0.734,0.266


## Assists

### prizepicks

In [5]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_assists')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()
league_avg_ast_ratio = league_df['AST_RATIO'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_ast = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_assists(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_def_rtg, league_avg_pace,
        league_avg_ast_ratio, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_ast % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_ast) + 1)
    elif target_ast % 1 == 0:
        prob_over_poisson = poisson.sf(target_ast, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_ast), int(target_ast) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_ast,
        'L-5': round(count_line_hits(player_df, target_ast, 'player_assists', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_ast, 'player_assists', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_ast, 'player_assists', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
assist_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
assist_df.to_csv(f'data/props/prizepicks/player_assists.csv', index=False)
assist_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Coby White,4.0,0.8,0.5,0.33,0.731,0.269
1,Austin Reaves,4.5,0.6,0.5,0.53,0.708,0.292
2,LeBron James,6.5,0.6,0.4,0.27,0.655,0.345
3,Immanuel Quickley,6.0,0.6,0.5,0.40,0.613,0.387
4,Jaylen Brown,5.0,0.6,0.5,0.33,0.610,0.390
5,Scottie Barnes,5.0,0.4,0.3,0.40,0.586,0.414
6,Toumani Camara,2.5,0.6,0.6,0.60,0.570,0.430
7,Payton Pritchard,4.5,0.4,0.4,0.47,0.550,0.450
8,Sandro Mamukelashvili,1.5,0.4,0.6,0.53,0.543,0.457
9,Cam Spencer,4.0,0.6,0.5,0.33,0.505,0.495


### underdog

In [6]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_assists')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()
league_avg_ast_ratio = league_df['AST_RATIO'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_ast = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_assists(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_def_rtg, league_avg_pace,
        league_avg_ast_ratio, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_ast % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_ast) + 1)
    elif target_ast % 1 == 0:
        prob_over_poisson = poisson.sf(target_ast, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_ast), int(target_ast) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_ast,
        'L-5': round(count_line_hits(player_df, target_ast, 'player_assists', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_ast, 'player_assists', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_ast, 'player_assists', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
assist_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
assist_df.to_csv(f'data/props/underdog/player_assists.csv', index=False)
assist_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Toumani Camara,2.5,0.6,0.6,0.60,0.570,0.430
1,Cameron Johnson,2.5,0.4,0.5,0.47,0.463,0.537
2,Anfernee Simons,2.5,0.4,0.5,0.53,0.422,0.578
3,Tyrese Maxey,6.5,0.4,0.4,0.53,0.361,0.639


# REBOUNDS

### prizepicks

In [7]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_rebounds')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_oreb = league_df['OREB_PCT'].mean()
league_avg_dreb = league_df['DREB_PCT'].mean()
league_avg_reb = league_df['REB_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_reb = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_rebounds(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_reb,
        league_avg_oreb, league_avg_dreb, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_reb % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_reb) + 1)
    elif target_reb % 1 == 0:
        prob_over_poisson = poisson.sf(target_reb, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_reb), int(target_reb) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_reb,
        'L-5': round(count_line_hits(player_df, target_reb, 'player_rebounds', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_reb, 'player_rebounds', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_reb, 'player_rebounds', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
rebound_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
rebound_df.to_csv(f'data/props/prizepicks/player_rebounds.csv', index=False)
rebound_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Austin Reaves,4.5,0.6,0.7,0.60,0.743,0.257
1,VJ Edgecombe,4.0,0.6,0.7,0.67,0.694,0.306
2,Matas Buzelis,5.5,0.6,0.4,0.47,0.659,0.341
3,Sion James,2.5,0.8,0.6,0.47,0.631,0.369
4,Lauri Markkanen,6.0,1.0,0.7,0.60,0.595,0.405
5,Donovan Clingan,10.0,0.8,0.7,0.47,0.586,0.414
6,Jaylen Brown,6.0,0.6,0.5,0.40,0.584,0.416
7,Cedric Coward,5.0,0.6,0.5,0.60,0.563,0.437
8,Quentin Grimes,3.5,0.6,0.4,0.40,0.563,0.437
9,Spencer Jones,3.5,0.6,0.6,0.40,0.555,0.445


### underdog

In [8]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_rebounds')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_oreb = league_df['OREB_PCT'].mean()
league_avg_dreb = league_df['DREB_PCT'].mean()
league_avg_reb = league_df['REB_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_reb = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_rebounds(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_reb,
        league_avg_oreb, league_avg_dreb, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_reb % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_reb) + 1)
    elif target_reb % 1 == 0:
        prob_over_poisson = poisson.sf(target_reb, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_reb), int(target_reb) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_reb,
        'L-5': round(count_line_hits(player_df, target_reb, 'player_rebounds', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_reb, 'player_rebounds', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_reb, 'player_rebounds', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
rebound_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
rebound_df.to_csv(f'data/props/underdog/player_rebounds.csv', index=False)
rebound_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Austin Reaves,4.5,0.6,0.7,0.60,0.743,0.257
1,Matas Buzelis,5.5,0.6,0.4,0.47,0.659,0.341
2,Quentin Grimes,3.5,0.6,0.4,0.40,0.563,0.437
3,Spencer Jones,3.5,0.6,0.6,0.40,0.555,0.445
4,Keyonte George,3.5,0.6,0.6,0.67,0.541,0.459
5,Josh Minott,4.5,0.2,0.3,0.40,0.485,0.515
6,Ja'Kobe Walter,3.5,0.6,0.4,0.27,0.462,0.538
7,Ryan Kalkbrenner,6.5,0.2,0.4,0.40,0.405,0.595
8,Bruce Brown,3.5,0.6,0.3,0.47,0.401,0.599
9,Ajay Mitchell,3.5,0.4,0.4,0.47,0.285,0.715


## Blocks

### prizepicks

In [9]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_blocks')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_blk = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_blocks(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_blk % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_blk) + 1)
    elif target_blk % 1 == 0:
        prob_over_poisson = poisson.sf(target_blk, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_blk), int(target_blk) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_blk,
        'L-5': round(count_line_hits(player_df, target_blk, 'player_blocks', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_blk, 'player_blocks', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_blk, 'player_blocks', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
    
blocks_df = pd.DataFrame(res)
blocks_df.to_csv(f'data/props/prizepicks/player_blocks.csv', index=False)
blocks_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%,IMPLIED_ODDS
0,Jordan Walsh,0.5,0.8,0.6,0.60,0.504,0.496,1.983
1,Josh Minott,0.5,0.6,0.4,0.33,0.374,0.626,2.672
2,Ryan Kalkbrenner,1.5,0.8,0.5,0.60,0.463,0.537,2.162
3,Brandon Miller,0.5,0.8,0.5,0.33,0.687,0.313,1.456
4,Jerami Grant,0.5,0.6,0.5,0.40,0.540,0.460,1.853
5,Santi Aldama,0.5,0.6,0.7,0.67,0.565,0.435,1.769
6,Toumani Camara,0.5,0.6,0.7,0.60,0.541,0.459,1.847
7,Jaylin Williams,0.5,0.6,0.3,0.33,0.345,0.655,2.899


### underdog

In [10]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_blocks')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_blk = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_blocks(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_blk % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_blk) + 1)
    elif target_blk % 1 == 0:
        prob_over_poisson = poisson.sf(target_blk, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_blk), int(target_blk) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_blk,
        'L-5': round(count_line_hits(player_df, target_blk, 'player_blocks', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_blk, 'player_blocks', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_blk, 'player_blocks', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
    
blocks_df = pd.DataFrame(res)
blocks_df.to_csv(f'data/props/underdog/player_blocks.csv', index=False)
blocks_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%,IMPLIED_ODDS
0,Ryan Kalkbrenner,1.5,0.8,0.5,0.60,0.463,0.537,2.162
1,Donovan Clingan,1.5,0.2,0.4,0.47,0.392,0.608,2.549


# STEALS

### prizepicks

In [11]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_steals')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_stl = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_steals(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_stl % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_stl) + 1)
    elif target_stl % 1 == 0:
        prob_over_poisson = poisson.sf(target_stl, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_stl), int(target_stl) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_stl,
        'L-5': round(count_line_hits(player_df, target_stl, 'player_steals', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_stl, 'player_steals', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_stl, 'player_steals', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
    steals_df = pd.DataFrame(res)
    steals_df.to_csv(f'data/props/prizepicks/player_steals.csv', index=False)
    steals_df.head(10)

### underdog

In [12]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_steals')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_stl = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_steals(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_stl % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_stl) + 1)
    elif target_stl % 1 == 0:
        prob_over_poisson = poisson.sf(target_stl, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_stl), int(target_stl) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_stl,
        'L-5': round(count_line_hits(player_df, target_stl, 'player_steals', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_stl, 'player_steals', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_stl, 'player_steals', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
    steals_df = pd.DataFrame(res)
    steals_df.to_csv(f'data/props/underdog/player_steals.csv', index=False)
    steals_df.head(10)

### prizepicks

In [13]:
## COMBO PROPS - All Categories

combo_categories = [
    'player_points_rebounds_assists',
    'player_points_rebounds',
    'player_points_assists',
    'player_rebounds_assists',
    'player_turnovers',
    'player_blocks_steals'
]

for category in combo_categories:
    print(f"\nProcessing {category}...")
    
    dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == category)]
    
    if dfs_data.empty:
        print(f"No data found for {category}")
        continue
    
    res = []
    PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))
    
    for _, row in PLAYERS.iterrows():
        PLAYER = row['NAME']
        target_line = row['LINE']

        player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
        if player_df.empty:
            continue

        res.append({
            'NAME': PLAYER,
            'LINE': target_line,
            'L-5': count_line_hits(player_df, target_line, category, [5])['L-5'],
            'L-10': count_line_hits(player_df, target_line, category, [10])['L-10'],
            'L-15': count_line_hits(player_df, target_line, category, [15])['L-15'],
        })
    
    if res:
        combo_df = pd.DataFrame(res).sort_values(by='L-5', ascending=False).reset_index(drop=True)
        combo_df.to_csv(f'data/props/prizepicks/{category}.csv', index=False)
        print(f"Saved {len(combo_df)} players for {category}")
        display(combo_df)
    else:
        print(f"No results for {category}")


Processing player_points_rebounds_assists...
Saved 63 players for player_points_rebounds_assists


,NAME,LINE,L-5,L-10,L-15
0,Jaylen Brown,41.5,1.0,0.7,0.67
1,Coby White,27.5,1.0,0.6,0.40
2,Jaylen Wells,17.5,1.0,0.7,0.53
3,Deni Avdija,41.0,1.0,0.7,0.60
4,Jordan Walsh,13.5,0.8,0.7,0.67
...,...,...,...,...,...
58,Brandon Miller,28.5,0.2,0.2,0.13
59,Ja'Kobe Walter,12.5,0.2,0.4,0.27
60,Brandon Ingram,33.5,0.2,0.4,0.47
61,Jalen Williams,34.5,0.0,0.0,0.00



Processing player_points_rebounds...
Saved 63 players for player_points_rebounds


,NAME,LINE,L-5,L-10,L-15
0,Jaylen Wells,16.0,1.0,0.6,0.47
1,Coby White,22.5,1.0,0.6,0.40
2,Jordan Walsh,12.5,0.8,0.7,0.67
3,VJ Edgecombe,14.5,0.8,0.8,0.80
4,Austin Reaves,27.5,0.8,0.9,0.80
...,...,...,...,...,...
58,Ajay Mitchell,22.5,0.0,0.0,0.20
59,Brandon Ingram,29.5,0.0,0.3,0.33
60,KJ Simpson,15.5,0.0,0.0,0.00
61,Kenrich Williams,11.5,0.0,0.0,0.00



Processing player_points_assists...
Saved 57 players for player_points_assists


...

,NAME,LINE,L-5,L-10,L-15
0,Sam Hauser,4.5,0.8,0.6,0.60
1,Quentin Grimes,7.5,0.8,0.5,0.47
2,Jaylen Brown,11.5,0.8,0.8,0.60
3,Draymond Green,11.5,0.8,0.7,0.53
4,Coby White,7.5,0.8,0.5,0.33
5,Matas Buzelis,7.5,0.8,0.5,0.47
6,Austin Reaves,9.0,0.8,0.8,0.73
7,Jared McCain,4.5,0.8,0.4,0.27
8,Zach Edey,13.5,0.8,0.5,0.33
9,Scottie Barnes,13.5,0.6,0.6,0.60



Processing player_turnovers...
Saved 13 players for player_turnovers


,NAME,LINE,L-5,L-10,L-15
0,Immanuel Quickley,1.5,0.8,0.5,0.53
1,Toumani Camara,1.5,0.6,0.7,0.73
2,Coby White,2.5,0.6,0.4,0.27
3,Draymond Green,2.5,0.6,0.8,0.80
4,Deandre Ayton,1.5,0.6,0.6,0.60
5,Payton Pritchard,1.5,0.4,0.3,0.40
6,Brandon Miller,2.5,0.4,0.2,0.13
7,Liam McNeeley,0.5,0.4,0.5,0.53
8,Will Richard,0.5,0.4,0.5,0.40
9,Keyonte George,3.5,0.4,0.5,0.33



Processing player_blocks_steals...
Saved 12 players for player_blocks_steals


,NAME,LINE,L-5,L-10,L-15
0,Brandon Miller,1.5,1.0,0.6,0.40
1,Peyton Watson,2.5,0.6,0.4,0.47
2,Joel Embiid,1.5,0.6,0.4,0.27
3,Paul George,1.5,0.6,0.5,0.33
4,Justin Edwards,0.5,0.6,0.7,0.67
5,Jaylin Williams,1.5,0.6,0.4,0.40
6,Jaylen Brown,1.5,0.4,0.4,0.40
7,Anfernee Simons,0.5,0.4,0.4,0.40
8,Draymond Green,1.5,0.4,0.5,0.47
9,Gabe Vincent,0.5,0.4,0.5,0.40


### underdog

In [14]:
## COMBO PROPS - All Categories

combo_categories = [
    'player_points_rebounds_assists',
    'player_points_rebounds',
    'player_points_assists',
    'player_rebounds_assists',
    'player_turnovers',
    'player_blocks_steals'
]

for category in combo_categories:
    print(f"\nProcessing {category}...")
    
    dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == category)]
    
    if dfs_data.empty:
        print(f"No data found for {category}")
        continue
    
    res = []
    PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))
    
    for _, row in PLAYERS.iterrows():
        PLAYER = row['NAME']
        target_line = row['LINE']

        player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
        if player_df.empty:
            continue

        res.append({
            'NAME': PLAYER,
            'LINE': target_line,
            'L-5': count_line_hits(player_df, target_line, category, [5])['L-5'],
            'L-10': count_line_hits(player_df, target_line, category, [10])['L-10'],
            'L-15': count_line_hits(player_df, target_line, category, [15])['L-15'],
        })
    
    if res:
        combo_df = pd.DataFrame(res).sort_values(by='L-5', ascending=False).reset_index(drop=True)
        combo_df.to_csv(f'data/props/underdog/{category}.csv', index=False)
        print(f"Saved {len(combo_df)} players for {category}")
        display(combo_df)
    else:
        print(f"No results for {category}")


Processing player_points_rebounds_assists...
Saved 58 players for player_points_rebounds_assists


,NAME,LINE,L-5,L-10,L-15
0,Deni Avdija,41.5,1.0,0.7,0.60
1,Jaylen Brown,41.5,1.0,0.7,0.67
2,Coby White,27.5,1.0,0.6,0.40
3,Spencer Jones,11.5,0.8,0.4,0.27
4,Tyrese Maxey,37.5,0.8,0.6,0.67
5,Quentin Grimes,21.5,0.8,0.7,0.67
6,Austin Reaves,32.5,0.8,0.8,0.73
7,Jordan Walsh,13.5,0.8,0.7,0.67
8,Cam Spencer,18.5,0.8,0.7,0.53
9,Immanuel Quickley,27.5,0.8,0.5,0.60



Processing player_points_rebounds...
Saved 28 players for player_points_rebounds


,NAME,LINE,L-5,L-10,L-15
0,Coby White,22.5,1.0,0.6,0.40
1,Deni Avdija,33.5,0.8,0.5,0.47
2,Derrick White,22.5,0.8,0.7,0.60
3,Austin Reaves,27.5,0.8,0.9,0.80
4,Jaylen Brown,35.5,0.8,0.6,0.60
5,Lauri Markkanen,30.5,0.6,0.5,0.47
6,Keyonte George,25.5,0.6,0.7,0.53
7,Deandre Ayton,23.5,0.6,0.6,0.53
8,Tyrese Maxey,31.5,0.6,0.5,0.60
9,Joel Embiid,23.5,0.6,0.4,0.27



Processing player_points_assists...
Saved 24 players for player_points_assists


...

,NAME,LINE,L-5,L-10,L-15
0,Jaylen Brown,11.5,0.8,0.8,0.60
1,Zach Edey,13.5,0.8,0.5,0.33
2,Matas Buzelis,7.5,0.8,0.5,0.47
3,Deandre Ayton,9.5,0.8,0.7,0.60
4,Quentin Grimes,7.5,0.8,0.5,0.47
5,Jamal Murray,10.5,0.6,0.7,0.73
6,Deni Avdija,15.5,0.6,0.5,0.40
7,Ryan Kalkbrenner,7.5,0.4,0.4,0.40
8,Shaedon Sharpe,7.5,0.4,0.5,0.47
9,LeBron James,12.5,0.4,0.3,0.20



Processing player_turnovers...
Saved 4 players for player_turnovers


,NAME,LINE,L-5,L-10,L-15
0,Draymond Green,2.5,0.6,0.8,0.80
1,Coby White,2.5,0.6,0.4,0.27
2,Brandon Miller,2.5,0.4,0.2,0.13
3,Keyonte George,3.5,0.4,0.5,0.33



Processing player_blocks_steals...
Saved 3 players for player_blocks_steals


,NAME,LINE,L-5,L-10,L-15
0,Peyton Watson,2.5,0.6,0.4,0.47
1,Zach Edey,2.5,0.6,0.5,0.33
2,Derrick White,2.5,0.2,0.4,0.53
